# MedZFS: Interactive Demo

This notebook demonstrates the MedZFS framework for zero-to-few-shot
medical image segmentation.

## Contents
1. Setup & Imports
2. Load Pre-Trained Model
3. Anatomical Knowledge Graph Visualization
4. Zero-Shot Segmentation Demo
5. Few-Shot Segmentation Demo
6. Prototype Visualization

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from models.medzfs import MedZFS
from data.anatomical_graphs import AnatomicalGraphBuilder
from utils.visualization import visualize_prediction, plot_shot_progression

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Anatomical Knowledge Graph

MedZFS uses anatomical knowledge graphs to constrain prototype
hallucination. Here we visualize the abdomen graph structure.

In [ ]:
# Build the abdominal anatomical graph
abd_graph = AnatomicalGraphBuilder.build_abdomen_graph()

print(f'Graph: {abd_graph.name}')
print(f'Nodes: {len(abd_graph.nodes)}')
print(f'Edges: {len(abd_graph.edges)}')
print(f'Relation types: {abd_graph.relation_types}')

print('\nNodes:')
for node in abd_graph.nodes:
    print(f'  - {node.name}: {node.description[:60]}...')

print('\nEdges (spatial):')
for edge in abd_graph.edges:
    if edge.relation == 'spatial':
        print(f'  - {edge.source} --[{edge.relation}]--> {edge.target}: {edge.description}')

## 2. Load Model

Load a pre-trained MedZFS checkpoint for inference.

In [ ]:
# Initialize model with default config
config = {
    'model': {
        'feature_dim': 512,
        'visual_encoder': {'pretrained': 'none', 'feature_blocks': [2, 3, 4]},
        'text_encoder': {'model_name': 'emilyalsentzer/Bio_ClinicalBERT', 'freeze': True},
        'graph_network': {'num_layers': 4, 'num_relation_types': 4},
        'hallucination': {'num_samples': 16},
        'prototype_mining': {'num_hard_prototypes': 8},
        'fusion': {'num_gnn_layers': 2, 'attention_heads': 8},
    }
}

model = MedZFS(config).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

# Uncomment to load a checkpoint:
# ckpt = torch.load('checkpoints/best_model.pth', map_location=device)
# model.load_state_dict(ckpt['model_state_dict'])

## 3. Zero-Shot Segmentation

Demonstrate segmentation without any labeled examples,
using only anatomical text descriptions.

In [ ]:
# Create a synthetic query image for demonstration
query = torch.randn(1, 3, 256, 256).to(device)

# Get graph data
edge_index, edge_type = abd_graph.get_full_edge_index()
edge_index = edge_index.to(device)
edge_type = edge_type.to(device)
node_descs = abd_graph.get_node_descriptions()

# Zero-shot prediction (no support set)
with torch.no_grad():
    outputs = model(
        query_image=query,
        class_description='A large, wedge-shaped organ in the upper right abdomen.',
        node_descriptions=node_descs,
        edge_index=edge_index,
        edge_type=edge_type,
    )

pred = outputs['prediction'].cpu().numpy()[0]
print(f'Prediction shape: {pred.shape}')
print(f'Hallucinated prototypes: {outputs["hallucinated_prototypes"].shape}')
print(f'Fused prototypes: {outputs["fused_prototypes"].shape}')

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(query.cpu().numpy()[0, 0], cmap='gray')
plt.title('Query Image')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(pred, cmap='jet', vmin=0, vmax=1)
plt.title('Zero-Shot Prediction')
plt.colorbar()
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Prototype Analysis

Analyze the hallucinated prototype distribution.

In [ ]:
# Visualize prototype distribution parameters
mu = outputs['mu'].cpu().numpy()[0]
log_var = outputs['log_var'].cpu().numpy()[0]
std = np.exp(0.5 * log_var)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(range(len(mu[:50])), mu[:50])
axes[0].set_title('Mean (first 50 dims)')
axes[0].set_xlabel('Dimension')

axes[1].bar(range(len(std[:50])), std[:50], color='orange')
axes[1].set_title('Std Dev (first 50 dims)')
axes[1].set_xlabel('Dimension')

protos = outputs['hallucinated_prototypes'].cpu().numpy()[0]  # (M, d)
similarity = protos @ protos.T
axes[2].imshow(similarity, cmap='viridis')
axes[2].set_title('Prototype Similarity Matrix')
axes[2].set_xlabel('Prototype')
axes[2].set_ylabel('Prototype')

plt.tight_layout()
plt.show()